In [ ]:
#   BUSQUEDA LOCAL (Escalada y Enfriamiento Simulado)

class Problema_Busqueda_Local(object):      #  (MEMORIZAR)
    """Clase abstracta para un problema de búsqueda local. Los problemas
    concretos habría que definirlos como subclases de esta clase,
    implementando genera_estado_inicial, genera_sucesor y valoración. Como
    atributo de dato, tendremos "mejor", que va a almacenar la función "menor
    que", o "mayor que" dependiendo de que se trate, respectivamente, de
    minimizar o maximizar."""     


    def __init__(self,mejor=lambda x,y: x < y ):
        self.mejor=mejor

    def genera_estado_inicial(self):
        """Genera, posiblemente con cierta componente aleatoria y heurística,
           un estado para empezar la búsqueda ."""
        pass
        
    def genera_sucesor(self, estado):
        """ Devuelve un estado "sucesor" del que recibe como
            entrada. Usualmente, esta función tendrá cierta componente
            aleatoria y heurística."""
        pass

    def valoracion(self, estado):
        """Devuelve la valoración de un estado. Es el valor a optimizar."""  
        pass

In [ ]:
#   ENFRIAMIENTO SIMULADO

def enfriamiento_simulado(problema, t_inicial, factor_descenso, n_enfriamientos, n_iteraciones):  # SIEMPRE ES IGUAL (MEMORIZAR)
    """Función que implementa el algoritmo de optimización mediante
       enfriamiento simulado. Recibe como entrada:
       - Un problema de búsqueda local (clase Problema_Busqueda_Local)
       - Una temperatura inicial
       - Un factor de descenso (para el programa de enfriamiento)
       - Un número total de enfriamientos 
       - Un número de iteraciones para cada temperatura"""

    actual = problema.genera_estado_inicial()
    valor_actual = problema.valoracion(actual)
    mejor = actual
    valor_mejor = valor_actual
    T = t_inicial

    for _ in range(n_enfriamientos):
        for _ in range(n_iteraciones):
            candidata = problema.genera_sucesor(actual)
            valor_candidata = problema.valoracion(candidata)

            if aceptar_e_s(valor_candidata, valor_actual, T, problema.mejor):   # Si aceptamos al candidato
                actual = candidata
                valor_actual = valor_candidata

                if problema.mejor(valor_actual, valor_mejor):   # Si este candidato es el mejor
                    mejor = actual
                    valor_mejor = valor_actual

        T *= factor_descenso
        
    return (mejor, valor_mejor)

In [ ]:
#   ALGORITMOS GENETICOS

class Problema_Genetico(object):
    """ Clase para representar un problema para que sea abordado mediante un
    algoritmo genético general. Consta de los siguientes atributos:
    - genes: lista de posibles genes en un cromosoma
    - longitud_cromosoma: la longitud de los cromosomas (todos son del mismo tamaño)
    - decodifica: método que recibe el genotipo (cromosoma) y devuelve el
      fenotipo (elemento del problema original que el cromosoma representa) 
    - fitness: método de valoración de los cromosomas (actúa sobre el
      genotipo)"""

    def __init__(self, genes, longitud_cromosoma, decodifica, fitness):
        self.genes = genes
        self.longitud_cromosoma = longitud_cromosoma
        self.decodifica = decodifica
        self.fitness = fitness






def algoritmo_genetico(problema_genetico, N, prop_padres, prob_mutar, nGen, k, mejor):
    """Algoritmo genético siguiendo el esquema descrito en el enunciado que, 
       además, va mostrando por pantalla estadísticas de las valoraciones de cada generación
       Devuelve el fenotipo del mejor cromosoma de la última generación, junto con su fitness.
       Recibe como entrada los siguientes argumentos:
       - problema_genetico: un objeto de la clase Problema_Genetico, con la representación del
         problema de optimización.
       - N: tamaño de cada generación
       - prop_padres: proporción de una generación que se usa como padres       por ejemplo: 0.8, si N = 100, entonces tenemos 80 padres
       - prob_mutar: probabilidad de mutación de un gen
       - nGen: número de generaciones 
       - k: número de cromosomas que intervienen en cada torneo
       - mejor: min o max, dependiendo si el problema es de minimización o de maximización
    """
    n_padres = round(N * prop_padres)                           # Selecciona el numero de cromosomas que van a ser padres
    n_padres = (n_padres if n_padres%2==0 else n_padres-1)      # Hace que el numero de padres sea par para que la funcion de cruzarlos no de error
    n_directos = N - n_padres            # Individuos de la generación actual que son seleccionados para pasar a la siguiente ronda directamente, sin cruzarse ni mutar

    generacion = problema_genetico.poblacion_inicial(N)
    
    for t in range(nGen):
        generacion = nueva_poblacion(problema_genetico, generacion, n_padres, n_directos, prob_mutar, k, mejor)
        mejor_cr = mejor(generacion, key = problema_genetico.fitness)
        mejor_fen = problema_genetico.decodifica(mejor_cr)
        print("Generacion: {0}. Media: {1}. Fitness mejor:{2}".format(t+1, media(problema_genetico.fitness, generacion, N),
                                                                      problema_genetico.fitness(mejor_cr)))        
    return (mejor_fen, problema_genetico.fitness(mejor_cr)) 

def media(fitness, generacion, N):
    return sum(map(fitness, generacion)) / N

In [ ]:
#   ESPACIO DE ESTADOS

class Problema(object):
    """Clase abstracta para un problema de espacio de estados. Los problemas
    concretos habría que definirlos como subclases de Problema, implementando
    acciones_aplicables, aplicar, y eventualmente __init__ y es_estado_final.
    Una vez hecho esto, se han de crear instancias de
    dicha subclase, que serán la entrada a los distintos algoritmos de
    resolución mediante búsqueda."""  

    def __init__(self, estado_inicial, estado_final=None):
        """El constructor de la clase especifica el estado inicial y
        puede que un estado_final, si es que es único. Las subclases podrían
        añadir otros argumentos"""
        
        self.estado_inicial = estado_inicial
        self.estado_final = estado_final

    def acciones_aplicables(self, estado):
        """Devuelve las acciones aplicables a un estado dado. Lo normal es
        que aquí se devuelva una lista, pero si hay muchas se podría devolver
        un iterador, ya que sería más eficiente."""
        pass

    def aplicar(self, estado, accion):
        """ Devuelve el estado resultante de aplicar accion a estado. Se
        supone que accion es aplicable a estado (es decir, debe ser una de las
        acciones que devuelve self.acciones_aplicables(estado)."""
        pass

    def es_estado_final(self, estado):
        """Devuelve True cuando estado es final. Por defecto, compara con el
        estado final, si éste se hubiera especificado al constructor. Si se da
        el caso de que no hubiera un único estado final, o se definiera
        mediante otro tipo de comprobación, habría que redefinir este método
        en la subclase.""" 
        return estado == self.estado_final
